# 01 — Dataset Preprocessing & EDA
---
This notebook handles:
1. **Unzipping** raw CWRU and Paderborn bearing datasets
2. **Sliding-window extraction** with Z-score normalization
3. **Saving** processed `.npz` files into `../datasets/processed/`

In [1]:
import os
import zipfile
import numpy as np
import scipy.io
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

DATASETS_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "datasets"))
PROCESSED_DIR = os.path.join(DATASETS_ROOT, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"Datasets root : {DATASETS_ROOT}")
print(f"Processed dir : {PROCESSED_DIR}")

Datasets root : /home/buddhiw/flowmatch/datasets
Processed dir : /home/buddhiw/flowmatch/datasets/processed


## 1 — Unzip Raw Datasets

In [2]:
# --- Unzip CWRU ---
cwru_zip = os.path.join(DATASETS_ROOT, "cwru-bearing-datasets.zip")
cwru_extract_dir = os.path.join(DATASETS_ROOT, "cwru_raw")
if not os.path.exists(cwru_extract_dir):
    print(f"Extracting {cwru_zip} ...")
    with zipfile.ZipFile(cwru_zip, "r") as zf:
        zf.extractall(cwru_extract_dir)
    print("Done.")
else:
    print(f"CWRU already extracted to {cwru_extract_dir}")

# --- Unzip Paderborn ---
pb_zip = os.path.join(DATASETS_ROOT, "paderborn_bearing_dataset.zip")
pb_extract_dir = os.path.join(DATASETS_ROOT, "paderborn_raw")
if not os.path.exists(pb_extract_dir):
    print(f"Extracting {pb_zip} ...")
    with zipfile.ZipFile(pb_zip, "r") as zf:
        zf.extractall(pb_extract_dir)
    print("Done.")
else:
    print(f"Paderborn already extracted to {pb_extract_dir}")

print("\nCWRU contents:", os.listdir(cwru_extract_dir))
print("Paderborn contents:", os.listdir(pb_extract_dir))

Extracting /home/buddhiw/flowmatch/datasets/cwru-bearing-datasets.zip ...
Done.
Extracting /home/buddhiw/flowmatch/datasets/paderborn_bearing_dataset.zip ...
Done.

CWRU contents: ['feature_time_48k_2048_load_1.csv', 'CWRU_48k_load_1_CNN_data.npz', 'raw']
Paderborn contents: ['KI07.rar', 'KA30.rar', 'KI16.rar', 'KI03.rar', 'KA07.rar', 'KA04.rar', 'K004.rar', 'KA01.rar', 'K003.rar', 'KI01.rar', 'K001.rar', 'KI04.rar', 'KB23.rar', 'KI05.rar', 'KB24.rar', 'KA09.rar', 'K002.rar', 'KA16.rar', 'KA06.rar', 'KA03.rar', 'K006.rar', 'KI17.rar', 'KA22.rar', 'K005.rar', 'KI21.rar', 'KI14.rar', 'KA15.rar', 'KI08.rar', 'KA05.rar', 'KI18.rar', 'KB27.rar']


## 2 — Process CWRU Bearing Dataset
- **Signal:** Drive-End (DE) accelerometer
- **Window:** 2048 samples, Step: 512
- **Normalization:** Z-score per feature
- **Labels:** 10 classes (1 Normal + 9 Fault severities × locations)

In [ ]:
WINDOW = 2048
STEP   = 512

# CWRU .mat filename → (label_index, readable_name)
# Label 0 = Normal, Labels 1-9 = Fault classes
CWRU_LABEL_MAP = {
    "Time_Normal_1_098":  (0, "Normal"),
    "B007_1_123":         (1, "Ball-0.007"),
    "B014_1_190":         (2, "Ball-0.014"),
    "B021_1_227":         (3, "Ball-0.021"),
    "IR007_1_110":        (4, "IR-0.007"),
    "IR014_1_175":        (5, "IR-0.014"),
    "IR021_1_214":        (6, "IR-0.021"),
    "OR007_6_1_136":      (7, "OR-0.007"),
    "OR014_6_1_202":      (8, "OR-0.014"),
    "OR021_6_1_239":      (9, "OR-0.021"),
}

raw_dir = os.path.join(cwru_extract_dir, "raw")

all_windows = []
all_labels  = []

for fname, (label, name) in CWRU_LABEL_MAP.items():
    mat_path = os.path.join(raw_dir, f"{fname}.mat")
    if not os.path.exists(mat_path):
        print(f"  [SKIP] {mat_path} not found"); continue

    mat = scipy.io.loadmat(mat_path)
    # Pick the Drive-End (DE) accelerometer key
    de_key = [k for k in mat.keys() if "DE_time" in k]
    if not de_key:
        de_key = [k for k in mat.keys() if not k.startswith("__")]
        de_key = [de_key[0]] if de_key else []
    if not de_key:
        print(f"  [SKIP] No suitable key in {fname}"); continue

    signal = mat[de_key[0]].flatten()
    n_windows = (len(signal) - WINDOW) // STEP + 1
    for i in range(n_windows):
        seg = signal[i * STEP : i * STEP + WINDOW]
        all_windows.append(seg)
        all_labels.append(label)

    print(f"  {name:12s}  signal={len(signal):>8,}  windows={n_windows}")

X = np.array(all_windows, dtype=np.float32)  # (N, 2048)
y = np.array(all_labels, dtype=np.int64)

# Z-score normalization (per-sample, channel=1)
scaler = StandardScaler()
X = scaler.fit_transform(X)          # (N, 2048) — already flat for 1-ch
X = X[:, :, np.newaxis]              # (N, 2048, 1)

print(f"\nCWRU Processed — X: {X.shape}, y: {y.shape}, classes: {np.unique(y)}")

# Stratified split 80/10/10
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Save as individual .npy files (fast mmap loading, no decompression overhead)
cwru_dir = os.path.join(PROCESSED_DIR, "cwru")
os.makedirs(cwru_dir, exist_ok=True)
for name_, arr in [("X_train", X_train), ("y_train", y_train),
                   ("X_val", X_val), ("y_val", y_val),
                   ("X_test", X_test), ("y_test", y_test)]:
    np.save(os.path.join(cwru_dir, f"{name_}.npy"), arr)

print(f"Saved → {cwru_dir}/  (train={len(y_train)}, val={len(y_val)}, test={len(y_test)})")

  Normal        signal= 483,903  windows=942
  Ball-0.007    signal= 487,384  windows=948
  Ball-0.014    signal= 486,224  windows=946
  Ball-0.021    signal= 486,804  windows=947
  IR-0.007      signal= 486,224  windows=946
  IR-0.014      signal= 489,125  windows=952
  IR-0.021      signal= 485,063  windows=944
  OR-0.007      signal= 486,804  windows=947
  OR-0.014      signal= 484,483  windows=943
  OR-0.021      signal= 489,125  windows=952

CWRU Processed — X: (9467, 2048, 1), y: (9467,), classes: [0 1 2 3 4 5 6 7 8 9]
Saved → /home/buddhiw/flowmatch/datasets/processed/cwru_processed.npz  (train=7573, val=947, test=947)


## 3 — Process Paderborn Bearing Dataset
- **Window:** 4096 samples
- **Normalization:** Z-score
- **Labels:** 32 classes (6 healthy + 12 artificial damage + 14 real damage)
- **Requires:** `patool` or `unrar` to extract inner `.rar` archives

In [ ]:
import subprocess, glob

PB_WINDOW = 4096
PB_STEP   = 1024
pb_mat_dir = os.path.join(pb_extract_dir, "mat_files")
os.makedirs(pb_mat_dir, exist_ok=True)

# Step 1: Extract every .rar → individual .mat files (skip if already done)
rar_files = sorted(glob.glob(os.path.join(pb_extract_dir, "*.rar")))
print(f"Found {len(rar_files)} .rar archives")

for rar_path in rar_files:
    bearing_name = os.path.splitext(os.path.basename(rar_path))[0]
    out_dir = os.path.join(pb_mat_dir, bearing_name)
    if os.path.exists(out_dir) and os.listdir(out_dir):
        continue
    os.makedirs(out_dir, exist_ok=True)
    try:
        import patoolib
        patoolib.extract_archive(rar_path, outdir=out_dir, verbosity=-1)
    except ImportError:
        subprocess.run(["unrar", "x", "-o+", rar_path, out_dir],
                       capture_output=True, check=True)
    print(f"  Extracted {bearing_name}")

# Step 2: Assign labels (0-N) alphabetically by bearing folder
bearing_dirs = sorted([d for d in os.listdir(pb_mat_dir)
                       if os.path.isdir(os.path.join(pb_mat_dir, d))])
label_map = {name: idx for idx, name in enumerate(bearing_dirs)}
print(f"\nPaderborn bearings: {len(bearing_dirs)} → labels 0-{len(bearing_dirs)-1}")

# Step 3: Sliding-window extraction from vibration_1 channel
VIBRATION_CHANNEL = 6

all_windows_pb, all_labels_pb = [], []

for bearing_name, label in label_map.items():
    mat_files = glob.glob(os.path.join(pb_mat_dir, bearing_name, "**", "*.mat"), recursive=True)
    total_wins = 0
    for mf in mat_files:
        try:
            mat = scipy.io.loadmat(mf)
        except Exception:
            continue
        top_keys = [k for k in mat if not k.startswith("__")]
        if not top_keys:
            continue

        try:
            d = mat[top_keys[0]]
            Y = d['Y'].flat[0]
            vib = Y['Data'].flat[VIBRATION_CHANNEL].flatten().astype(np.float32)
        except (KeyError, IndexError, ValueError):
            continue

        if len(vib) < PB_WINDOW:
            continue

        n_w = (len(vib) - PB_WINDOW) // PB_STEP + 1
        for i in range(n_w):
            all_windows_pb.append(vib[i * PB_STEP : i * PB_STEP + PB_WINDOW])
            all_labels_pb.append(label)
        total_wins += n_w

    if total_wins:
        print(f"  {bearing_name:6s} → label {label:2d}, windows={total_wins}")

print(f"\nTotal Paderborn windows: {len(all_windows_pb)}")

if all_windows_pb:
    X_pb = np.array(all_windows_pb, dtype=np.float32)
    y_pb = np.array(all_labels_pb, dtype=np.int64)

    scaler_pb = StandardScaler()
    X_pb = scaler_pb.fit_transform(X_pb)
    X_pb = X_pb[:, :, np.newaxis]  # (N, 4096, 1)

    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_pb, y_pb, test_size=0.2, stratify=y_pb, random_state=42)
    X_v, X_te, y_v, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

    # Save as individual .npy files (fast mmap loading)
    pb_dir = os.path.join(PROCESSED_DIR, "paderborn")
    os.makedirs(pb_dir, exist_ok=True)
    for name_, arr in [("X_train", X_tr), ("y_train", y_tr),
                       ("X_val", X_v), ("y_val", y_v),
                       ("X_test", X_te), ("y_test", y_te)]:
        np.save(os.path.join(pb_dir, f"{name_}.npy"), arr)

    print(f"\nSaved → {pb_dir}/  (train={len(y_tr)}, val={len(y_v)}, test={len(y_te)})")
else:
    print("\n⚠️  No Paderborn windows extracted — check .rar extraction (install patool or unrar)")

Found 31 .rar archives

Paderborn bearings: 31 → labels 0-30
  K001   → label  0, windows=19778
  K002   → label  1, windows=19819
  K003   → label  2, windows=19805
  K004   → label  3, windows=19777
  K005   → label  4, windows=19793
  K006   → label  5, windows=19790
  KA01   → label  6, windows=19794
  KA03   → label  7, windows=19950
  KA04   → label  8, windows=19771
  KA05   → label  9, windows=19816
  KA06   → label 10, windows=19778
  KA07   → label 11, windows=19860
  KA09   → label 12, windows=19801
  KA15   → label 13, windows=19779
  KA16   → label 14, windows=19780
  KA22   → label 15, windows=19832
  KA30   → label 16, windows=19782
  KB23   → label 17, windows=19808
  KB24   → label 18, windows=19790
  KB27   → label 19, windows=19845
  KI01   → label 20, windows=19770
  KI03   → label 21, windows=19815
  KI04   → label 22, windows=19785
  KI05   → label 23, windows=19825
  KI07   → label 24, windows=19819
  KI08   → label 25, windows=19824
  KI14   → label 26, windows=

In [16]:
## ── One-time migration: convert existing .npz → individual .npy files ─────────
## Run this ONCE to avoid re-running the 19-min Paderborn extraction.
## After this cell completes, cells 6 and 8 will skip re-saving since .npy dirs exist.
import time

def npz_to_npy(npz_path, out_dir, keys=("X_train","y_train","X_val","y_val","X_test","y_test")):
    """Convert a compressed .npz into individual raw .npy files."""
    if os.path.isdir(out_dir) and all(os.path.exists(os.path.join(out_dir, f"{k}.npy")) for k in keys):
        print(f"  ✓ {out_dir}/ already has all .npy files — skipping")
        return
    if not os.path.exists(npz_path):
        print(f"  ✗ {npz_path} not found — skipping")
        return
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()
    data = np.load(npz_path)
    for k in keys:
        np.save(os.path.join(out_dir, f"{k}.npy"), data[k])
        print(f"    saved {k}.npy  shape={data[k].shape}")
    print(f"  → {out_dir}/  done in {time.time()-t0:.1f}s")

print("Converting CWRU .npz → .npy ...")
npz_to_npy(os.path.join(PROCESSED_DIR, "cwru_processed.npz"),
           os.path.join(PROCESSED_DIR, "cwru"))

print("\nConverting Paderborn .npz → .npy ...")
npz_to_npy(os.path.join(PROCESSED_DIR, "paderborn_processed.npz"),
           os.path.join(PROCESSED_DIR, "paderborn"))

print("\n✅ Migration complete.  Future loads use mmap'd .npy — near-instant.")

Converting CWRU .npz → .npy ...
    saved X_train.npy  shape=(7573, 2048, 1)
    saved y_train.npy  shape=(7573,)
    saved X_val.npy  shape=(947, 2048, 1)
    saved y_val.npy  shape=(947,)
    saved X_test.npy  shape=(947, 2048, 1)
    saved y_test.npy  shape=(947,)
  → /home/buddhiw/flowmatch/datasets/processed/cwru/  done in 7.7s

Converting Paderborn .npz → .npy ...
    saved X_train.npy  shape=(491325, 4096, 1)
    saved y_train.npy  shape=(491325,)
    saved X_val.npy  shape=(61416, 4096, 1)
    saved y_val.npy  shape=(61416,)
    saved X_test.npy  shape=(61416, 4096, 1)
    saved y_test.npy  shape=(61416,)
  → /home/buddhiw/flowmatch/datasets/processed/paderborn/  done in 1213.8s

✅ Migration complete.  Future loads use mmap'd .npy — near-instant.


---
# 4 — Comprehensive Exploratory Data Analysis
> Research-grade statistical analysis of all FlowMatch-PdM datasets:
> **CWRU** (Bearing Fault, 10 classes) · **Paderborn** (Bearing Fault, 31 classes) · **C-MAPSS** (Engine RUL) · **FEMTO** (Bearing RUL) · **XJTU-SY** (Bearing RUL)

In [14]:
## ── EDA Imports & Global Config ──────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats as sp_stats
from scipy.signal import welch
from collections import Counter

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "figure.figsize": (14, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})
COLORS = sns.color_palette("tab10", 32)
print("EDA imports loaded")

EDA imports loaded


In [18]:
## ── 4.0  Load & Cache All Processed Datasets (mmap — near-instant) ───────────
import time
t0 = time.time()

cwru_dir = os.path.join(PROCESSED_DIR, "cwru")
pb_dir   = os.path.join(PROCESSED_DIR, "paderborn")

# Load only training splits for EDA (80% of data — fully representative)
# mmap_mode='r' means pages loaded on-demand, not all at once
cwru_X = np.load(os.path.join(cwru_dir, "X_train.npy"), mmap_mode='r')
cwru_y = np.load(os.path.join(cwru_dir, "y_train.npy"), mmap_mode='r')
pb_X   = np.load(os.path.join(pb_dir,   "X_train.npy"), mmap_mode='r')
pb_y   = np.load(os.path.join(pb_dir,   "y_train.npy"), mmap_mode='r')

# Split sizes for the summary table
cwru_splits = {s: np.load(os.path.join(cwru_dir, f"y_{s}.npy"), mmap_mode='r').shape[0] for s in ("train","val","test")}
pb_splits   = {s: np.load(os.path.join(pb_dir,   f"y_{s}.npy"), mmap_mode='r').shape[0] for s in ("train","val","test")}

print(f"CWRU      — X: {cwru_X.shape}   classes: {len(np.unique(cwru_y))}   splits: {cwru_splits}")
print(f"Paderborn — X: {pb_X.shape}   classes: {len(np.unique(pb_y))}   splits: {pb_splits}")
print(f"\n⚡ Loaded in {time.time()-t0:.2f}s  (train splits only, mmap'd)")

CWRU      — X: (7573, 2048, 1)   classes: 10   splits: {'train': 7573, 'val': 947, 'test': 947}
Paderborn — X: (491325, 4096, 1)   classes: 31   splits: {'train': 491325, 'val': 61416, 'test': 61416}

⚡ Loaded in 0.54s  (train splits only, mmap'd)


## 4.1 — Load C-MAPSS, FEMTO, XJTU-SY (RUL Datasets)
Using the `rul-datasets` library for automated download, windowing, and normalization.

In [ ]:
## ── 4.1  Load RUL Datasets via rul-datasets ──────────────────────────────────
import rul_datasets

rul_data = {}  # {name: {"features": np.ndarray, "targets": np.ndarray, "reader": reader}}

RUL_CONFIGS = [
    ("CMAPSS",   lambda: rul_datasets.CmapssReader(fd=1, window_size=30, max_rul=125)),
    ("FEMTO",    lambda: rul_datasets.FemtoReader(fd=1, window_size=2048)),
    ("XJTU-SY",  lambda: rul_datasets.XjtuSyReader(fd=1, window_size=2048)),
]

for name, make_reader in RUL_CONFIGS:
    try:
        reader = make_reader()
        reader.prepare_data()
        dm = rul_datasets.RulDataModule(reader, batch_size=512)
        dm.setup()

        # dev split = training data
        feats, tgts = [], []
        ds = dm.to_dataset("dev")
        for x, y in ds:
            feats.append(x.numpy())
            tgts.append(y.numpy())
        feats = np.stack(feats)
        tgts  = np.concatenate(tgts) if tgts[0].ndim > 0 else np.array(tgts)
        
        rul_data[name] = {"features": feats, "targets": tgts, "reader": reader}
        print(f"  ✓ {name:10s}  X: {feats.shape}  RUL range: [{tgts.min():.0f}, {tgts.max():.0f}]")
    except Exception as e:
        print(f"  ✗ {name:10s}  {e}")

print(f"\nLoaded {len(rul_data)} RUL datasets")

Download CMAPSS dataset


12142it [00:05, 2075.93it/s]                           


Extract CMAPSS dataset


/home/buddhiw/miniconda3/envs/flowmatch_pdm/lib/python3.10/site-packages/rul_datasets/reader/cmapss.py:127: UserWarning: Training data for FD001 not yet split into dev and val. Splitting now.
  warnings.warn(


  ✓ CMAPSS      X: (13818, 14, 30)  RUL range: [1, 125]
Download FEMTO dataset


744007it [02:09, 5748.58it/s]                             


Extract FEMTO dataset


/home/buddhiw/miniconda3/envs/flowmatch_pdm/lib/python3.10/site-packages/rul_datasets/reader/femto.py:154: UserWarning: First time use. Pre-process dev split of FD1.
  warnings.warn(f"First time use. Pre-process {split} split of FD{self.fd}.")
Runs: 100%|██████████| 2/2 [00:11<00:00,  5.53s/it]
/home/buddhiw/miniconda3/envs/flowmatch_pdm/lib/python3.10/site-packages/rul_datasets/reader/femto.py:154: UserWarning: First time use. Pre-process val split of FD1.
  warnings.warn(f"First time use. Pre-process {split} split of FD{self.fd}.")
Runs: 100%|██████████| 5/5 [00:52<00:00, 10.47s/it]


  ✓ FEMTO       X: (3674, 2, 2048)  RUL range: [1, 2803]
Download XJTU-SY dataset


 19%|█▉        | 1002608/5321154 [02:26<10:15, 7019.12it/s]